# Predicting whether a constituency changes party

Binary logistic regression, evaluated separately on **2005, 2010, 2015, 2017 and 2019**. Each model learns only from earlier elections in `TEST_TRAIN/train.csv`; the 2024 test file is not used.

`seat_changed = 1` when `winner != previous_winner`. Here “incumbent” means the constituency's previous winning party, not the national governing party stored in the source `incumbent` column.

Party-specific predictors are mapped to fixed roles using previous results: incumbent, strongest non-incumbent (the contesting party), third party, and fourth party. Normally the contesting party is the previous runner-up. If the recorded previous winner is not the largest share, it still defines the incumbent, and challengers are ranked among the remaining parties. Ties use the fixed order `con, lib, lab, natSW`. The same mapping is used for previous shares, projections, polling and previous national shares; projections never determine the roles.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, precision_score, recall_score,
                             brier_score_loss, accuracy_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "TEST_TRAIN" / "train.csv").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run this notebook from inside the repository.")
raw = pd.read_csv(ROOT / "TEST_TRAIN" / "train.csv")
EVALUATION_YEARS = [2005, 2010, 2015, 2017, 2019]
PARTIES = ["con", "lib", "lab", "natSW"]
ROLES = ["incumbent", "contesting_party", "third_party", "fourth_party"]
display(raw.groupby("election").size().rename("available_rows").to_frame())

,available_rows
election,
1987,633
1992,634
1997,641
2001,641
2005,628
2010,632
2015,632
2017,632
2019,632


## Eligibility and role-based features

Previous winners labelled `oth` cannot be mapped because there is no corresponding previous share column. Exclude those rows, missing previous winners, missing outcome labels, and incomplete previous party shares (needed to rank challengers). **Keep current `oth` winners** when the previous winner is supported: these are valid seat changes. The audit below reports coverage by election.

`natSW` is the dataset's combined SNP/Plaid category. There are no polling or projected-share fields for this category. Missing role-specific values are imputed using only the training fold's medians, with missingness indicators. Existing projections are used as provided, without clipping or recomputing them. The upstream SQL uses previous national *seat* shares in its projection formula, so these should not be interpreted as calibrated vote forecasts.

Current-election shares, majority and winner are never predictors. The previous winning party’s last-election vote share and region are retained. Party identities and the national governing party are categorical predictors.

In [2]:
previous_columns = [f"previous_{p}_share" for p in PARTIES]
known_previous = raw["previous_winner"].isin(PARTIES)
complete_previous = raw[previous_columns].notna().all(axis=1)
known_outcome = raw["winner"].notna()
eligible = known_previous & complete_previous & known_outcome
coverage = raw.assign(
    eligible=eligible,
    unsupported_previous=~known_previous,
    incomplete_previous=~complete_previous,
    missing_outcome=~known_outcome,
).groupby("election").agg(
    available=("eligible", "size"), included=("eligible", "sum"),
    unsupported_previous=("unsupported_previous", "sum"),
    incomplete_previous=("incomplete_previous", "sum"),
    missing_outcome=("missing_outcome", "sum"),
)
coverage["excluded"] = coverage["available"] - coverage["included"]
display(coverage)  # Exclusion reasons can overlap.
data = raw.loc[eligible].copy().reset_index(drop=True)


def engineer_features(frame):
    """Use only pre-election fields; preserve each party's role across families."""
    shares = frame[previous_columns].to_numpy(dtype=float)
    incumbent_index = pd.Index(PARTIES).get_indexer(frame["previous_winner"])
    if (incumbent_index < 0).any() or not np.isfinite(shares).all():
        raise ValueError("Role mapping needs supported previous winners and complete shares.")
    ranked = np.argsort(-shares, axis=1, kind="stable")
    challengers = ranked[ranked != incumbent_index[:, None]].reshape(-1, 3)
    role_indices = np.column_stack([incumbent_index, challengers])
    families = {
        "share": previous_columns,
        "projected_share": ["projected_con_share", "projected_lib_share", "projected_lab_share", None],
        "polling": ["con_polling", "lib_polling", "lab_polling", None],
        "previous_national_share": [f"previous_nat_{p}_share" for p in PARTIES],
    }
    features = pd.DataFrame(index=frame.index)
    for role_number, role in enumerate(ROLES):
        features[f"{role}_party"] = np.asarray(PARTIES)[role_indices[:, role_number]]
        for suffix, columns in families.items():
            values = np.column_stack([
                frame[col].to_numpy(dtype=float) if col else np.full(len(frame), np.nan)
                for col in columns
            ])
            features[f"{role}_{suffix}"] = values[np.arange(len(frame)), role_indices[:, role_number]]
    features["previous_share_gap"] = features["incumbent_share"] - features["contesting_party_share"]
    features["projected_share_gap"] = features["incumbent_projected_share"] - features["contesting_party_projected_share"]
    features["previous_winning_party_last_election_vote_share"] = frame["previous_winning_party_last_election_vote_share"]
    features["country/region"] = frame["country/region"]
    features["national_governing_party"] = frame["incumbent"]
    return features

X = engineer_features(data)
y = data["winner"].ne(data["previous_winner"]).astype(int).rename("seat_changed")
assert X["incumbent_party"].equals(data["previous_winner"].rename("incumbent_party"))
assert X["incumbent_party"].ne(X["contesting_party_party"]).all()
# Changing the outcome cannot change the features.
pd.testing.assert_frame_equal(X, engineer_features(data.drop(columns=["winner"])))
display(pd.concat([data[["constituency_name", "election", "previous_winner", "winner"]],
                   X.iloc[:, :10], y], axis=1).head(10))

,available,included,unsupported_previous,incomplete_previous,missing_outcome,excluded
election,,,,,,
1987,633,633,0,0,0,0
1992,634,633,1,1,0,1
1997,641,550,91,91,0,91
2001,641,639,2,2,0,2
2005,628,626,2,2,0,2
2010,632,631,1,1,0,1
2015,632,630,2,0,0,2
2017,632,628,4,1,0,4
2019,632,629,2,3,0,3


,constituency_name,election,previous_winner,winner,incumbent_party,incumbent_share,incumbent_projected_share,incumbent_polling,incumbent_previous_national_share,contesting_party_party,contesting_party_share,contesting_party_projected_share,contesting_party_polling,contesting_party_previous_national_share,seat_changed
0,Aberavon,1987,lab,lab,lab,0.588,0.599076,0.341250,0.330174,lib,0.203,0.381353,0.214688,0.036335,0
1,Aberdeen North,1987,lab,lab,lab,0.470,0.481076,0.341250,0.330174,lib,0.247,0.425353,0.214688,0.036335,0
2,Aberdeen South,1987,con,lab,con,0.389,0.188703,0.426875,0.627172,lab,0.299,0.310076,0.341250,0.330174,1
3,Aldershot,1987,con,con,con,0.554,0.353703,0.426875,0.627172,lib,0.338,0.516353,0.214688,0.036335,0
4,Aldridge-Brownhills,1987,con,con,con,0.507,0.306703,0.426875,0.627172,lab,0.249,0.260076,0.341250,0.330174,0
5,Altrincham & Sale,1987,con,con,con,0.525,0.324703,0.426875,0.627172,lib,0.299,0.477353,0.214688,0.036335,0
6,Alyn & Deeside,1987,lab,lab,lab,0.403,0.414076,0.341250,0.330174,con,0.372,0.171703,0.426875,0.627172,0
7,Amber Valley,1987,con,con,con,0.417,0.216703,0.426875,0.627172,lab,0.353,0.364076,0.341250,0.330174,0
8,Angus East,1987,con,natSW,con,0.441,0.240703,0.426875,0.627172,natSW,0.360,NaN,NaN,0.006319,1
9,Argyll & Bute,1987,con,lib,con,0.386,0.185703,0.426875,0.627172,lib,0.275,0.453353,0.214688,0.036335,1


## Expanding-window evaluation

Fit a new preprocessing pipeline and logistic regression for every evaluation year. Medians, scaling parameters and categorical encodings are fitted exclusively on earlier elections. Hyperparameters are fixed (`C=1`, no class weighting), and the classification threshold is **0.5**, with seat change as the positive class.

ROC-AUC and Brier score use predicted probabilities; precision, recall and accuracy use thresholded predictions. Higher is better except for Brier score. Results describe the eligible constituencies shown in the coverage table. The hold/no-change baseline accuracy is included because seat changes may be uncommon.

`predicted_change_rate` is the proportion of evaluated seats predicted to change at the 0.5 threshold, regardless of whether those predictions are correct: `(true positives + false positives) / n_evaluation`.

In [3]:
categorical_columns = X.select_dtypes(include=["object", "string"]).columns.tolist()
numeric_columns = X.columns.difference(categorical_columns).tolist()


def make_model():
    numeric = Pipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True)),
        ("scale", StandardScaler()),
    ])
    categorical = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("encode", OneHotEncoder(handle_unknown="ignore")),
    ])
    return Pipeline([
        ("preprocess", ColumnTransformer([
            ("numeric", numeric, numeric_columns),
            ("categorical", categorical, categorical_columns),
        ])),
        ("classifier", LogisticRegression(C=1.0, solver="lbfgs", max_iter=3000)),
    ])

models = {}
rows = []
prediction_frames = []
for year in EVALUATION_YEARS:
    train_mask = data["election"] < year
    test_mask = data["election"] == year
    if not train_mask.any() or not test_mask.any():
        raise ValueError(f"Missing training or evaluation rows for {year}.")
    assert data.loc[train_mask, "election"].max() < year
    if y.loc[train_mask].nunique() != 2:
        raise ValueError(f"Training set for {year} needs both classes.")
    model = make_model()
    model.fit(X.loc[train_mask], y.loc[train_mask])
    models[year] = model
    positive_column = list(model.named_steps["classifier"].classes_).index(1)
    probabilities = model.predict_proba(X.loc[test_mask])[:, positive_column]
    predictions = (probabilities >= 0.5).astype(int)
    actual = y.loc[test_mask]
    rows.append({
        "election": year,
        "training_years": ", ".join(map(str, sorted(data.loc[train_mask, "election"].unique()))),
        "n_train": int(train_mask.sum()), "n_evaluation": int(test_mask.sum()),
        "change_rate": actual.mean(),
        "predicted_change_rate": predictions.mean(),
        "roc_auc": roc_auc_score(actual, probabilities) if actual.nunique() == 2 else np.nan,
        "precision": precision_score(actual, predictions, zero_division=0),
        "recall": recall_score(actual, predictions, zero_division=0),
        "brier_score": brier_score_loss(actual, probabilities),
        "accuracy": accuracy_score(actual, predictions),
        "always_hold_accuracy": (actual == 0).mean(),
    })
    prediction_frames.append(data.loc[test_mask, ["constituency_name", "election", "previous_winner", "winner"]]
        .assign(seat_changed=actual, change_probability=probabilities, predicted_change=predictions))

results = pd.DataFrame(rows).set_index("election")
predictions = pd.concat(prediction_frames, ignore_index=True)
display(results[["n_train", "n_evaluation", "roc_auc", "precision", "recall", "brier_score", "accuracy", "predicted_change_rate"]].round(4))
display(results[["training_years", "change_rate", "always_hold_accuracy"]])
assert len(predictions) == results["n_evaluation"].sum()
assert predictions["change_probability"].between(0, 1).all()

,n_train,n_evaluation,roc_auc,precision,recall,brier_score,accuracy,predicted_change_rate
election,,,,,,,,
2005,2455,626,0.8765,0.6000,0.0526,0.0647,0.9105,0.0080
2010,3081,631,0.8554,0.5205,0.8018,0.1242,0.8352,0.2710
2015,3712,630,0.7484,0.5325,0.3761,0.1273,0.8349,0.1222
2017,4342,628,0.8533,0.3448,0.1538,0.0928,0.8822,0.0462
2019,4970,629,0.9023,0.4507,0.4267,0.0857,0.8696,0.1129


,training_years,change_rate,always_hold_accuracy
election,,,
2005,"1987, 1992, 1997, 2001",0.091054,0.908946
2010,"1987, 1992, 1997, 2001, 2005",0.175911,0.824089
2015,"1987, 1992, 1997, 2001, 2005, 2010",0.173016,0.826984
2017,"1987, 1992, 1997, 2001, 2005, 2010, 2015",0.103503,0.896497
2019,"1987, 1992, 1997, 2001, 2005, 2010, 2015, 2017",0.119237,0.880763


The fitted pipelines are available in `models[year]`, the metric table in `results`, and constituency-level probabilities in `predictions`. Precision/recall are reported as zero if their denominator is zero; ROC-AUC is undefined (`NaN`) if an evaluation year contains only one class.

## First-stage confusion matrices: hold or change

These matrices evaluate the **binary seat-change model** on all eligible seats in each held-out election, using the existing **0.5 change-probability threshold**. Rows show actual outcomes; columns show predicted outcomes. Each election has its own matrix, reported as seat counts.

- Actual hold, predicted hold: **true negatives** — correctly predicted holds.
- Actual hold, predicted change: **false positives** — holds incorrectly flagged as changes.
- Actual change, predicted hold: **false negatives** — missed seat changes.
- Actual change, predicted change: **true positives** — correctly detected changes.

For each election, recall is the bottom-right count divided by the sum of the actual-change row: `TP / (FN + TP)`. This measures how many actual changes the first stage detects, independently of destination-party prediction.


In [4]:
from sklearn.metrics import confusion_matrix

change_confusion_matrices = {}
for year in EVALUATION_YEARS:
    election_predictions = predictions.loc[predictions["election"].eq(year)]
    counts = confusion_matrix(
        election_predictions["seat_changed"],
        election_predictions["predicted_change"],
        labels=[0, 1],
    )
    assert counts.sum() == len(election_predictions)
    tn, fp, fn, tp = counts.ravel()
    assert np.isclose(tp / (tp + fn) if tp + fn else 0, results.loc[year, "recall"])
    change_confusion_matrices[year] = pd.DataFrame(
        counts,
        index=pd.Index(["Hold (0)", "Change (1)"], name="Actual outcome"),
        columns=pd.Index(["Hold (0)", "Change (1)"], name="Predicted outcome"),
    )

change_confusion_table = pd.concat(change_confusion_matrices, names=["Election"])
display(change_confusion_table)


Predicted outcome        Hold (0)  Change (1)
Election Actual outcome                      
2005     Hold (0)             567           2
         Change (1)            54           3
2010     Hold (0)             438          82
         Change (1)            22          89
2015     Hold (0)             485          36
         Change (1)            68          41
2017     Hold (0)             544          19
         Change (1)            55          10
2019     Hold (0)             515          39
         Change (1)            43          32

## Conditional destination model: who wins if the seat changes?

For each evaluation election, fit a separate multinomial logistic regression using **only changed seats from earlier elections**. Reuse the pre-election feature engineering and pipeline specification, but fit preprocessing afresh on that smaller training set. The target is the winning challenger role: `contesting_party`, `third_party`, `fourth_party`, or `oth`. Map role probabilities back to party labels for each constituency. This construction gives the incumbent zero conditional probability by definition, rather than allowing an impossible “change to the incumbent”.

`oth` is an aggregate category, and `natSW` retains the dataset's combined SNP/Plaid definition. A destination role absent from a training fold receives zero probability: the model cannot learn an unseen class. The class-count table makes this visible. No current outcome is used as an input feature.

Produce conditional probabilities for **every eligible seat**, since the actual change is unknown at prediction time. Evaluate this stage only on seats that actually changed in the held-out election. These are conditional metrics, not end-to-end performance.

For incumbent party $i$, combine the two stages using:

$$P(W=i\mid X)=1-P(C=1\mid X)$$
$$P(W=p\mid X)=P(C=1\mid X)P(W=p\mid C=1,X),\quad p
e i.$$

Use the original change probabilities, not thresholded labels. The resulting party probabilities sum to one; normalization alone does not establish calibration.

In [5]:
from sklearn.metrics import log_loss

DESTINATION_ROLES = ["contesting_party", "third_party", "fourth_party", "oth"]
PARTY_LABELS = PARTIES + ["oth"]

# Outcomes define training labels and evaluation subsets only.
destination_target = pd.Series(pd.NA, index=data.index, dtype="object")
for role in DESTINATION_ROLES[:-1]:
    matches = y.eq(1) & data["winner"].eq(X[f"{role}_party"])
    destination_target.loc[matches] = role
destination_target.loc[y.eq(1) & data["winner"].eq("oth")] = "oth"
if destination_target.loc[y.eq(1)].isna().any():
    raise ValueError("Some changed-seat winners cannot be mapped to a destination role.")


def party_probability_metrics(actual, probabilities):
    """Multiclass Brier is the mean sum of squared errors across parties (0–2)."""
    actual = np.asarray(actual)
    predicted = np.asarray(PARTY_LABELS)[probabilities.argmax(axis=1)]
    one_hot = (actual[:, None] == np.asarray(PARTY_LABELS)[None, :]).astype(float)
    # sklearn log_loss expects probability columns in sorted label order.
    sorted_columns = np.argsort(PARTY_LABELS)
    return {
        "accuracy": accuracy_score(actual, predicted),
        "log_loss": log_loss(actual, probabilities[:, sorted_columns],
                             labels=sorted(PARTY_LABELS)),
        "multiclass_brier": np.square(probabilities - one_hot).sum(axis=1).mean(),
    }


destination_models = {}
destination_training_counts = []
conditional_metric_rows = []
combined_metric_rows = []
conditional_frames = []
combined_frames = []

for year in EVALUATION_YEARS:
    changed_train = data["election"].lt(year) & y.eq(1)
    evaluation = data["election"].eq(year)
    evaluation_data = data.loc[evaluation]
    evaluation_features = X.loc[evaluation]
    train_target = destination_target.loc[changed_train]
    if train_target.nunique() < 2:
        raise ValueError(f"{year}: at least two destination classes are needed.")
    assert data.loc[changed_train, "election"].max() < year
    assert y.loc[changed_train].eq(1).all()
    destination_training_counts.append({
        "election": year,
        **train_target.value_counts().reindex(DESTINATION_ROLES, fill_value=0).to_dict(),
    })
    destination_model = make_model()
    destination_model.fit(X.loc[changed_train], train_target)
    destination_models[year] = destination_model
    role_probabilities = pd.DataFrame(
        destination_model.predict_proba(evaluation_features),
        index=evaluation_data.index,
        columns=destination_model.named_steps["classifier"].classes_,
    ).reindex(columns=DESTINATION_ROLES, fill_value=0.0)

    conditional = np.zeros((len(evaluation_data), len(PARTY_LABELS)))
    row_indices = np.arange(len(evaluation_data))
    for role in DESTINATION_ROLES[:-1]:
        party_indices = pd.Index(PARTY_LABELS).get_indexer(evaluation_features[f"{role}_party"])
        conditional[row_indices, party_indices] += role_probabilities[role].to_numpy()
    conditional[:, PARTY_LABELS.index("oth")] = role_probabilities["oth"].to_numpy()
    incumbent_indices = pd.Index(PARTY_LABELS).get_indexer(evaluation_data["previous_winner"])
    assert np.allclose(conditional[row_indices, incumbent_indices], 0)
    assert np.allclose(conditional.sum(axis=1), 1)

    # The first stage was already fitted on all eligible earlier seats.
    change_class = list(models[year].named_steps["classifier"].classes_).index(1)
    change_probability = models[year].predict_proba(evaluation_features)[:, change_class]
    combined = conditional * change_probability[:, None]
    combined[row_indices, incumbent_indices] = 1 - change_probability
    assert np.allclose(combined.sum(axis=1), 1)
    assert np.allclose(combined[row_indices, incumbent_indices], 1 - change_probability)
    assert ((combined >= 0) & (combined <= 1)).all()

    identifiers = evaluation_data[["constituency_name", "election", "previous_winner", "winner"]].copy()
    conditional_frame = identifiers.copy()
    combined_frame = identifiers.assign(change_probability=change_probability)
    for column, party in enumerate(PARTY_LABELS):
        conditional_frame[f"p_{party}_given_change"] = conditional[:, column]
        combined_frame[f"p_{party}"] = combined[:, column]
    conditional_frame["predicted_destination"] = np.asarray(PARTY_LABELS)[conditional.argmax(axis=1)]
    combined_frame["predicted_winner"] = np.asarray(PARTY_LABELS)[combined.argmax(axis=1)]
    conditional_frames.append(conditional_frame)
    combined_frames.append(combined_frame)

    actually_changed = y.loc[evaluation].eq(1).to_numpy()
    conditional_metrics = (
        party_probability_metrics(evaluation_data.loc[actually_changed, "winner"], conditional[actually_changed])
        if actually_changed.any()
        else dict.fromkeys(["accuracy", "log_loss", "multiclass_brier"], np.nan)
    )
    conditional_metric_rows.append({
        "election": year, "n_changed_train": int(changed_train.sum()),
        "n_changed_evaluation": int(actually_changed.sum()), **conditional_metrics,
    })
    combined_metric_rows.append({
        "election": year, "n_evaluation": len(evaluation_data),
        **party_probability_metrics(evaluation_data["winner"], combined),
        "changed_seat_accuracy": (
            accuracy_score(
                evaluation_data.loc[actually_changed, "winner"],
                combined_frame.loc[actually_changed, "predicted_winner"],
            ) if actually_changed.any() else np.nan
        ),
    })

conditional_party_probabilities = pd.concat(conditional_frames, ignore_index=True)
combined_party_probabilities = pd.concat(combined_frames, ignore_index=True)
conditional_results = pd.DataFrame(conditional_metric_rows).set_index("election")
combined_results = pd.DataFrame(combined_metric_rows).set_index("election")
destination_training_counts = pd.DataFrame(destination_training_counts).set_index("election")


### Destination-role counts in earlier changed-seat training data

This table shows the examples available to train the conditional destination model for each evaluation election. The row labelled `2019`, for example, counts eligible seats that changed party in elections **before 2019**; it does not include the 2019 outcomes. Each count represents a constituency-election observation, so a constituency can contribute more than once across earlier elections.

The columns identify the role of the party that actually gained the seat, using the pre-election role mapping:

- `contesting_party`: the strongest non-incumbent by previous vote share.
- `third_party`: the next strongest non-incumbent.
- `fourth_party`: the remaining supported non-incumbent.
- `oth`: a winner outside the four supported party categories.

These are role counts, not counts for fixed parties: Labour, for example, can occupy different challenger roles in different constituencies. Each row sums to `n_changed_train` in the next table. The balance of counts shows how much evidence the model has for each destination; a zero means that destination is absent from that training fold and receives zero conditional probability.


In [6]:
display(destination_training_counts)


,contesting_party,third_party,fourth_party,oth
election,,,,
2005,254,18,1,4
2010,304,23,1,6
2015,411,25,1,8
2017,496,37,11,10
2019,554,44,11,10


### Conditional destination performance: evaluated only on actual changed seats

This table asks: **given that a seat changed party, how well did the second stage identify its new winner?** Each row evaluates a model trained on earlier elections against only the eligible seats that actually changed in the named held-out election. Although the model produces conditional probabilities for every eligible seat, unchanged seats are excluded from these metrics.

- `n_changed_train`: the number of earlier changed-seat observations used to fit the destination model.
- `n_changed_evaluation`: the number of actual changed seats used to score it in the evaluation election.
- `accuracy`: the fraction of those seats where the party with the highest conditional probability was the actual winner. For example, `0.80` means 80% were correctly identified.
- `log_loss`: a score for the probability assigned to the actual winner. Lower is better; a confident wrong prediction is penalized heavily.
- `multiclass_brier`: the mean, across evaluated seats, of the sum of squared differences between the party probabilities and the actual outcome (1 for the winning party, 0 for every other party). Lower is better, with 0 perfect and a theoretical maximum of 2.

These scores assess destination prediction under the condition that a change occurred. They do not measure whether the first stage correctly anticipated that change. Rare destinations can be difficult to learn; an unseen destination has zero probability, which scikit-learn clips internally when calculating log loss. Rows with fewer changed seats also give less evidence about performance.


In [7]:
display(conditional_results.round(4))


,n_changed_train,n_changed_evaluation,accuracy,log_loss,multiclass_brier
election,,,,,
2005,277,57,0.8772,0.6089,0.2226
2010,334,111,0.9459,0.1825,0.0866
2015,445,109,0.6514,1.6948,0.5846
2017,554,65,0.8308,0.7951,0.2791
2019,619,75,0.9067,0.2333,0.1290


### Combined party prediction performance: evaluated on all eligible seats

This table evaluates the complete two-stage prediction of **which party wins each seat**, including both holds and changes. For each held-out election, the incumbent receives `1 - change_probability`; each challenger receives the first stage's change probability multiplied by its conditional destination probability. The predicted winner is the party with the largest resulting probability, without applying the binary 0.5 change threshold.

`n_evaluation` counts all eligible seats in that election. `accuracy` is the fraction whose final predicted winner is correct, while `log_loss` and `multiclass_brier` score the full combined party probabilities using the definitions above. Higher accuracy and lower probability losses indicate better performance.

Unlike the conditional table, these results reflect errors from both stages: estimating whether the incumbent loses and identifying the party that replaces it. The two tables use different evaluation populations, so their scores are not a direct like-for-like comparison; overall accuracy can be strongly influenced by the number of seats that remain with the incumbent. The multiclass Brier score also uses a different scale from the earlier binary change-model Brier score.

`changed_seat_accuracy` is the fraction of **actually changed seats** where the final combined predicted winner matches the actual winner. Predicting an incumbent hold for one of these seats counts as incorrect. It uses the same changed-seat subset as the conditional table, but scores the complete two-stage prediction. It is `NaN` if no eligible seats changed in that election.


In [8]:
display(combined_results.round(4))


,n_evaluation,accuracy,log_loss,multiclass_brier,changed_seat_accuracy
election,,,,,
2005,626,0.9105,0.2901,0.1328,0.0526
2010,631,0.8399,0.4814,0.2496,0.7387
2015,630,0.8270,0.7611,0.2672,0.3211
2017,628,0.8822,0.4057,0.1893,0.1385
2019,629,0.8712,0.2913,0.1735,0.3733


The fitted conditional models remain available in `destination_models[year]`. Constituency-level results are stored in `conditional_party_probabilities` (party probabilities given a change) and `combined_party_probabilities` (final party probabilities) for all five evaluation elections. The `winner` column is retained for inspection and evaluation only.
